# Fraud Pattern Analysis

## Project Overview
Analyze transaction data to identify patterns, spikes, segments, and transaction characteristics associated with fraudulent activity. This is a **data analysis** project — we explore fraud patterns descriptively rather than building a fraud detection model.

## Learning Objectives
- Understand the characteristics of fraudulent vs legitimate transactions
- Identify temporal, geographic, and amount-based fraud patterns
- Segment fraud by transaction type, channel, and merchant category
- Derive actionable rules and insights for fraud prevention

## Problem Statement
A payments company is experiencing increasing fraud losses. They need to understand *what fraud looks like* — timing, amounts, channels, merchant categories — so they can design better detection rules and allocate investigation resources.

## Why This Project Matters
Global fraud losses exceed $30 billion annually. Understanding fraud patterns enables rule-based detection, anomaly flagging, and resource prioritization — critical even before deploying ML models.

## Dataset Overview
Synthetic transaction data: ~20K transactions with ~3% fraud rate. Includes amount, merchant category, channel, time features, customer age group, and fraud label.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by published payment fraud research
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_txn = 20000
fraud_rate = 0.03
n_fraud = int(n_txn * fraud_rate)
n_legit = n_txn - n_fraud

categories = ['Retail', 'Online Shopping', 'Travel', 'Dining', 'Gas Station',
              'ATM Withdrawal', 'Wire Transfer', 'Subscription']
cat_weights_legit = [0.25, 0.20, 0.08, 0.15, 0.12, 0.08, 0.04, 0.08]
cat_weights_fraud = [0.10, 0.30, 0.15, 0.05, 0.05, 0.10, 0.20, 0.05]

channels = ['POS', 'Online', 'ATM', 'Mobile App', 'Phone']
ch_weights_legit = [0.35, 0.30, 0.10, 0.15, 0.10]
ch_weights_fraud = [0.10, 0.40, 0.15, 0.10, 0.25]

dates = pd.date_range('2024-01-01', '2024-12-31', freq='h')

# Legitimate transactions
legit_amounts = np.clip(np.random.lognormal(3.5, 1.0, n_legit), 1, 5000).round(2)
legit_cats = np.random.choice(categories, n_legit, p=cat_weights_legit)
legit_channels = np.random.choice(channels, n_legit, p=ch_weights_legit)
legit_dates = np.random.choice(dates, n_legit)
legit_hours = pd.to_datetime(legit_dates).hour

# Fraudulent transactions — different patterns
fraud_amounts = np.clip(np.random.lognormal(5.0, 1.5, n_fraud), 10, 50000).round(2)
fraud_cats = np.random.choice(categories, n_fraud, p=cat_weights_fraud)
fraud_channels = np.random.choice(channels, n_fraud, p=ch_weights_fraud)
# Fraud peaks at night (11pm-4am)
fraud_date_pool = dates[dates.hour.isin([23, 0, 1, 2, 3, 4]) | (np.random.random(len(dates)) < 0.3)]
fraud_dates = np.random.choice(fraud_date_pool, n_fraud)

age_groups = ['18-25', '26-35', '36-45', '46-55', '55+']
age_weights = [0.15, 0.30, 0.25, 0.20, 0.10]

df = pd.concat([
    pd.DataFrame({'Amount': legit_amounts, 'Category': legit_cats, 'Channel': legit_channels,
                  'Timestamp': legit_dates, 'IsFraud': 0,
                  'AgeGroup': np.random.choice(age_groups, n_legit, p=age_weights)}),
    pd.DataFrame({'Amount': fraud_amounts, 'Category': fraud_cats, 'Channel': fraud_channels,
                  'Timestamp': fraud_dates, 'IsFraud': 1,
                  'AgeGroup': np.random.choice(age_groups, n_fraud, p=age_weights)}),
], ignore_index=True)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.day_name()
df['Month'] = df['Timestamp'].dt.month
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.insert(0, 'TxnID', [f'TXN{i:06d}' for i in range(len(df))])

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate: {df["IsFraud"].mean():.2%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nFraud distribution:\n{df["IsFraud"].value_counts()}')
print(f'\nAmount range: ${df["Amount"].min():.2f} - ${df["Amount"].max():,.2f}')
print(f'Date range: {df["Timestamp"].min()} to {df["Timestamp"].max()}')
print(f'Categories: {df["Category"].nunique()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount distribution: fraud vs legit
df[df['IsFraud']==0]['Amount'].hist(ax=axes[0,0], bins=50, alpha=0.6, label='Legitimate', edgecolor='black')
df[df['IsFraud']==1]['Amount'].hist(ax=axes[0,0], bins=50, alpha=0.6, label='Fraud', edgecolor='black', color='red')
axes[0,0].set_title('Transaction Amount Distribution')
axes[0,0].legend()
axes[0,0].set_xlabel('Amount ($)')

# Fraud rate by category
fraud_by_cat = df.groupby('Category')['IsFraud'].mean().sort_values(ascending=True)
fraud_by_cat.plot.barh(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Fraud Rate by Merchant Category')
axes[0,1].set_xlabel('Fraud Rate')

# Hourly fraud pattern
hourly = df.groupby(['Hour', 'IsFraud']).size().unstack(fill_value=0)
hourly_rate = hourly[1] / hourly.sum(axis=1)
hourly_rate.plot(ax=axes[1,0], marker='o', color='red')
axes[1,0].set_title('Fraud Rate by Hour of Day')
axes[1,0].set_ylabel('Fraud Rate')
axes[1,0].set_xlabel('Hour')

# Fraud by channel
fraud_by_ch = df.groupby('Channel')['IsFraud'].mean().sort_values(ascending=True)
fraud_by_ch.plot.barh(ax=axes[1,1], edgecolor='black', color='steelblue')
axes[1,1].set_title('Fraud Rate by Channel')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Fraud Amount Analysis

In [ ]:
fraud_txns = df[df['IsFraud'] == 1]
legit_txns = df[df['IsFraud'] == 0]

print('Amount Statistics:')
print(f'  Legitimate — Mean: ${legit_txns["Amount"].mean():.2f}, Median: ${legit_txns["Amount"].median():.2f}')
print(f'  Fraud      — Mean: ${fraud_txns["Amount"].mean():.2f}, Median: ${fraud_txns["Amount"].median():.2f}')
print(f'  Fraud/Legit ratio (mean): {fraud_txns["Amount"].mean() / legit_txns["Amount"].mean():.1f}x')

# Amount buckets
df['AmountBucket'] = pd.cut(df['Amount'], bins=[0, 50, 200, 500, 1000, 5000, 50000],
                             labels=['<$50', '$50-200', '$200-500', '$500-1K', '$1K-5K', '$5K+'])
bucket_fraud = df.groupby('AmountBucket')['IsFraud'].agg(['count', 'sum', 'mean']).round(4)
bucket_fraud.columns = ['Total', 'Fraud', 'FraudRate']
print('\nFraud Rate by Amount Bucket:')
print(bucket_fraud)

## Temporal Fraud Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_fraud = df.groupby('DayOfWeek')['IsFraud'].mean().reindex(dow_order)
dow_fraud.plot.bar(ax=axes[0], edgecolor='black', color='salmon')
axes[0].set_title('Fraud Rate by Day of Week')
axes[0].set_ylabel('Fraud Rate')
axes[0].tick_params(axis='x', rotation=45)

# Monthly trend
monthly = df.groupby('Month').agg(total=('IsFraud','count'), fraud=('IsFraud','sum'))
monthly['rate'] = monthly['fraud'] / monthly['total']
axes[1].bar(monthly.index, monthly['total'], alpha=0.3, label='Total Txns', color='steelblue')
ax2 = axes[1].twinx()
ax2.plot(monthly.index, monthly['rate'], color='red', marker='o', label='Fraud Rate')
axes[1].set_title('Monthly Volume & Fraud Rate')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Transaction Count')
ax2.set_ylabel('Fraud Rate')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'temporal_patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Category × Channel Fraud Heatmap

In [ ]:
cat_ch_fraud = df.groupby(['Category', 'Channel'])['IsFraud'].mean().unstack(fill_value=0)
plt.figure(figsize=(10, 6))
sns.heatmap(cat_ch_fraud, annot=True, fmt='.3f', cmap='YlOrRd')
plt.title('Fraud Rate: Category × Channel')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'category_channel.png'), dpi=100, bbox_inches='tight')
plt.show()

## Financial Impact

In [ ]:
total_fraud_loss = fraud_txns['Amount'].sum()
avg_fraud_amount = fraud_txns['Amount'].mean()
total_volume = df['Amount'].sum()

print(f'Total transaction volume: ${total_volume:,.2f}')
print(f'Total fraud losses: ${total_fraud_loss:,.2f}')
print(f'Fraud as % of volume: {total_fraud_loss/total_volume:.2%}')
print(f'Average fraud amount: ${avg_fraud_amount:,.2f}')
print(f'Average legit amount: ${legit_txns["Amount"].mean():,.2f}')
print(f'\nTop fraud categories by loss:')
cat_loss = fraud_txns.groupby('Category')['Amount'].sum().sort_values(ascending=False)
for cat, loss in cat_loss.items():
    print(f'  {cat}: ${loss:,.2f}')

## Fraud Detection Rules (Heuristic)

In [ ]:
# Test simple rule-based flags
df['Flag_HighAmount'] = df['Amount'] > 1000
df['Flag_NightTime'] = df['Hour'].isin([23, 0, 1, 2, 3, 4])
df['Flag_OnlineWire'] = df['Channel'].isin(['Online', 'Phone']) & df['Category'].isin(['Wire Transfer', 'Online Shopping'])
df['Flag_Any'] = df['Flag_HighAmount'] | df['Flag_NightTime'] | df['Flag_OnlineWire']

for flag in ['Flag_HighAmount', 'Flag_NightTime', 'Flag_OnlineWire', 'Flag_Any']:
    flagged = df[df[flag]]
    precision = flagged['IsFraud'].mean() if len(flagged) > 0 else 0
    recall = flagged['IsFraud'].sum() / df['IsFraud'].sum() if df['IsFraud'].sum() > 0 else 0
    print(f'{flag}:')
    print(f'  Flagged: {len(flagged)} ({len(flagged)/len(df):.1%}), Precision: {precision:.1%}, Recall: {recall:.1%}')

## Interpretation and Business Insight
- **Fraud amounts are significantly larger** than legitimate transactions (~3-5x higher), making amount thresholds a useful first filter
- **Night-time transactions** (11pm-4am) have elevated fraud rates — fraudsters exploit off-hours when monitoring is lighter
- **Wire Transfer** and **Online Shopping** categories have the highest fraud rates — these are easier to execute remotely
- **Phone** and **Online** channels are the riskiest — no physical card present
- Simple heuristic rules can catch a meaningful fraction of fraud, though with notable false positive rates
- The Category × Channel heatmap reveals specific high-risk combinations for targeted monitoring

## Limitations
- Synthetic data with injected patterns — real fraud is more diverse and adversarial
- No customer behavior history (velocity checks, device fingerprinting)
- Binary fraud label doesn't distinguish fraud types (account takeover, card not present, synthetic identity)
- Rule-based analysis misses complex non-linear patterns that ML models can catch

## How to Improve This Project
- Use real anonymized transaction data for more realistic patterns
- Add velocity features (transactions per hour, new merchant flags)
- Build anomaly detection models (Isolation Forest, Autoencoders)
- Analyze fraud networks and organized fraud rings
- Add cost-based analysis: balancing false positive costs vs fraud losses

## Production Considerations
- Deploy real-time scoring with sub-second latency requirements
- Implement feedback loops: confirmed fraud labels improve future detection
- Monitor rule performance over time — fraudsters adapt to known rules
- Regulatory compliance: PCI DSS, anti-money laundering requirements

## Common Mistakes
- Evaluating fraud detection on accuracy (misleading when fraud is 3% of transactions)
- Not considering the cost of false positives (blocked legitimate transactions erode trust)
- Training models on balanced datasets then deploying on imbalanced production data
- Ignoring concept drift — fraud patterns evolve continuously

## Mini Challenge / Exercises
1. Design a rule that achieves >50% recall with precision >15%. What trade-offs did you make?
2. Calculate the dollar-weighted fraud rate (fraud losses / total volume) by category.
3. Find the hour with the highest fraud rate. Is it also the hour with the most fraud volume?
4. Create a "risk score" (0-10) based on amount bucket + hour + channel. How well does it separate fraud?

## Final Summary / Key Takeaways
- Fraud analysis starts with understanding what fraud looks like before building models
- Amount, timing, channel, and category are the four pillars of transaction-level fraud analysis
- Simple rules provide a useful baseline but suffer from precision-recall trade-offs
- The financial impact of fraud is concentrated in specific categories and channels
- Effective fraud prevention combines rule-based detection, ML models, and human investigation